## Week 5 EXERCISE: RAG for an Indie Game Publisher

### Knowledge base Q&A over Pixel Forge Games internal docs

This notebook implements a **RAG (Retrieval Augmented Generation)** assistant over a local knowledge base:
- **Published titles** – catalogue of games and platforms
- **Submission guidelines** – how to submit a game, what we look for
- **Revenue share & terms** – splits, payment schedule, advances
- **Platform requirements** – Steam, Epic, Nintendo Switch, Xbox
- **Team & contacts** – who to email for submissions, contracts, QA, marketing

**Pipeline:** Load markdown from `game-publisher-kb/` → chunk → embed → Chroma → retriever → LLM with context → Gradio chat.

**Run the notebook from the `week5` folder** so paths `game-publisher-kb` and `game_publisher_db` resolve correctly.

### Part A: Load documents and split into chunks

In [31]:
import os
import glob
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [32]:
MODEL = "openai/gpt-4o-mini"
DB_NAME = "game_publisher_db"
KNOWLEDGE_BASE_DIR = "game-publisher-kb"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

In [ ]:
# Load all .md files from each subfolder and tag with doc_type (folder name)
folders = glob.glob(os.path.join(KNOWLEDGE_BASE_DIR, "*"))
folders = [f for f in folders if os.path.isdir(f)]

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents from {KNOWLEDGE_BASE_DIR}")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

### Part B: Embed and store in Chroma

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
)
print(f"Vectorstore created: {vectorstore._collection.count()} chunks in {DB_NAME}")

### Part C: RAG – retriever, system prompt, and answer function

In [36]:
# Reconnect to the persisted store (use this cell when you only want to run RAG without re-ingesting)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    temperature=0,
    model=MODEL,
    streaming=True,
)

In [37]:
SYSTEM_PROMPT_TEMPLATE = """
You are a helpful assistant for Pixel Forge Games, an indie game publisher.
You answer questions using only the following context from the company's internal knowledge base.
If the context does not contain the answer, say so. Do not make up details.
Be concise and accurate.

Context:
{context}
"""

In [38]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    full = ""
    for chunk in llm.stream([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ]):
        if chunk.content:
            full += chunk.content
            yield full
    if not full:
        yield "No response generated."

In [ ]:
# answer_question is a generator when streaming; consume it to get the full reply
full_reply = ""
for part in answer_question("What is the revenue share on Steam for developers?", []):
    full_reply = part
print(full_reply)

### Gradio UI

In [ ]:
with gr.Blocks(title="Pixel Forge Games – Publisher KB Q&A") as app:
    gr.Markdown(
        "Ask questions about **published titles**, **submission guidelines**, **revenue share & terms**, **platform requirements** (Steam, Epic, Switch, Xbox), or **team contacts**. "
        "Answers are based on the internal knowledge base."
    )
    gr.ChatInterface(answer_question)

In [ ]:
app.launch()